# Sales ETL Notebook


## 1. Setup
Import libraries and configure paths, outputs, and logging.

In [10]:
import json
import logging
import sys
from pathlib import Path
import os

import pandas as pd
import numpy as np

In [11]:
# Paths and output directories
BASE_DIR = Path.cwd().resolve()
RAW_DIR = BASE_DIR / "Data" / "Raw" # Assuming raw data is provided in this relative path"
OUT_DIR = BASE_DIR / "Data" / "Processed"
LOG_DIR = BASE_DIR / "docs"

SALES_FILE = RAW_DIR / "Sales.json"
FORECAST_FILE = RAW_DIR / "forecast.json"

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)


In [12]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    handlers=[
        logging.FileHandler(LOG_DIR / "etl_run.log", mode="w"),
        logging.StreamHandler(sys.stdout),
    ],
)
log = logging.getLogger("orion_etl")

## 2. Extract
Load the raw JSON files into pandas DataFrames.

In [13]:
def extract(path: Path, name: str) -> pd.DataFrame:
    log.info(f"EXTRACT | Loading {name} from {path}")
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    df = pd.DataFrame(data)
    log.info(f"EXTRACT | {name}: {len(df):,} rows, {len(df.columns)} columns")
    return df

## 3. Clean
Define cleaning routines for Sales and Forecast data.

In [14]:
def clean_sales(df: pd.DataFrame) -> pd.DataFrame:
    log.info("CLEAN | Sales - starting cleaning routine")
    initial_rows = len(df)

    dupes = df.duplicated().sum()
    df = df.drop_duplicates().reset_index(drop=True)
    log.info(f"CLEAN | Sales - removed {dupes:,} duplicate rows ({initial_rows:,} -> {len(df):,})")

    str_cols = df.select_dtypes(include="object").columns
    for c in str_cols:
        df[c] = df[c].astype(str).where(df[c].notna(), df[c])
        df[c] = df[c].apply(lambda x: x.strip() if isinstance(x, str) else x)
    log.info(f"CLEAN | Sales - trimmed whitespace on {len(str_cols)} text columns")

    if "Color" in df.columns and "Subcategory" in df.columns:
        mismatches = (df["Color"] != df["Subcategory"]).sum()
        if mismatches == 0:
            df = df.drop(columns=["Color"])
            log.info("CLEAN | Sales - dropped redundant 'Color' column")
        else:
            log.warning(f"CLEAN | Sales - 'Color' differs from 'Subcategory' in {mismatches} rows")

    df["OrderDate"] = pd.to_datetime(df["OrderDate"], format="%m/%d/%Y", errors="coerce")
    bad_dates = df["OrderDate"].isna().sum()
    if bad_dates:
        log.warning(f"CLEAN | Sales - {bad_dates} rows had unparseable OrderDate and were dropped")
        df = df.dropna(subset=["OrderDate"])

    invalid_qty = df[df["Quantity"] <= 0]
    if len(invalid_qty):
        log.warning(f"CLEAN | Sales - removing {len(invalid_qty)} rows with Quantity <= 0")
        df = df[df["Quantity"] > 0]

    invalid_price = df[df["Net Price"] <= 0]
    if len(invalid_price):
        log.warning(f"CLEAN | Sales - removing {len(invalid_price)} rows with Net Price <= 0")
        df = df[df["Net Price"] > 0]

    for col in ["Name", "Education", "Occupation"]:
        if col in df.columns:
            nulls = df[col].isna().sum()
            df[col] = df[col].fillna("Unknown")
            log.info(f"CLEAN | Sales - filled {nulls:,} nulls in '{col}' with 'Unknown'")

    df["ProductKey"] = df["ProductKey"].astype(int)
    df["CustomerKey"] = df["CustomerKey"].astype(int)
    df["Quantity"] = df["Quantity"].astype(int)
    df["Net Price"] = df["Net Price"].astype(float).round(4)

    log.info(f"CLEAN | Sales - finished. Final row count: {len(df):,}")
    return df.reset_index(drop=True)


def clean_forecast(df: pd.DataFrame) -> pd.DataFrame:
    log.info("CLEAN | Forecast - starting cleaning routine")
    initial_rows = len(df)

    dupes = df.duplicated().sum()
    df = df.drop_duplicates().reset_index(drop=True)
    log.info(f"CLEAN | Forecast - removed {dupes} duplicate rows ({initial_rows} -> {len(df)})")

    for c in ["CountryRegion", "Brand"]:
        if c in df.columns:
            df[c] = df[c].astype(str).str.strip()

    df["Forecast"] = df["Forecast"].astype(float).round(2)
    df["Year"] = df["Year"].astype(int)

    invalid = df[df["Forecast"] <= 0]
    if len(invalid):
        log.warning(f"CLEAN | Forecast - {len(invalid)} rows with Forecast <= 0 found, retained for review")

    log.info(f"CLEAN | Forecast - finished. Final row count: {len(df)}")
    return df

## 4. Transform
Build dimension and fact tables for the star schema.

In [15]:
def build_dim_product(sales: pd.DataFrame) -> pd.DataFrame:
    cols = ["ProductKey", "Product Name", "Brand", "Category", "Subcategory"]
    dim = sales[cols].drop_duplicates(subset="ProductKey").reset_index(drop=True)
    return dim.rename(columns={"Product Name": "ProductName"})


def build_dim_customer(sales: pd.DataFrame) -> pd.DataFrame:
    cols = ["CustomerKey", "Customer Code", "Name", "Education", "Occupation",
            "Continent", "City", "State", "CountryRegion"]
    dim = sales[cols].drop_duplicates(subset="CustomerKey").reset_index(drop=True)
    return dim.rename(columns={"Customer Code": "CustomerCode"})


def build_dim_geography(sales: pd.DataFrame, forecast: pd.DataFrame) -> pd.DataFrame:
    geo_sales = sales[["Continent", "CountryRegion"]].drop_duplicates()
    geo_fc = forecast[["CountryRegion"]].drop_duplicates()
    geo = geo_sales.merge(geo_fc, on="CountryRegion", how="outer")
    geo["Continent"] = geo["Continent"].fillna("Unknown")
    geo = geo.drop_duplicates(subset="CountryRegion").reset_index(drop=True)
    geo.insert(0, "GeographyKey", range(1, len(geo) + 1))
    return geo


def build_dim_brand(sales: pd.DataFrame, forecast: pd.DataFrame) -> pd.DataFrame:
    brands = pd.Index(sales["Brand"].unique()).union(forecast["Brand"].unique())
    dim = pd.DataFrame({"Brand": sorted(brands)})
    dim.insert(0, "BrandKey", range(1, len(dim) + 1))
    return dim


def build_dim_date(sales: pd.DataFrame, forecast: pd.DataFrame) -> pd.DataFrame:
    min_date = sales["OrderDate"].min()
    max_date = sales["OrderDate"].max()

    fc_years = forecast["Year"].unique()
    fc_min = pd.Timestamp(year=int(min(fc_years)), month=1, day=1)
    fc_max = pd.Timestamp(year=int(max(fc_years)), month=12, day=31)

    start = min(min_date, fc_min)
    end = max(max_date, fc_max)

    dates = pd.date_range(start=start, end=end, freq="D")
    dim = pd.DataFrame({"Date": dates})
    dim["DateKey"] = dim["Date"].dt.strftime("%Y%m%d").astype(int)
    dim["Year"] = dim["Date"].dt.year
    dim["Quarter"] = dim["Date"].dt.quarter
    dim["QuarterName"] = "Q" + dim["Quarter"].astype(str) + " " + dim["Year"].astype(str)
    dim["Month"] = dim["Date"].dt.month
    dim["MonthName"] = dim["Date"].dt.strftime("%B")
    dim["MonthShort"] = dim["Date"].dt.strftime("%b")
    dim["YearMonth"] = dim["Date"].dt.strftime("%Y-%m")
    dim["Day"] = dim["Date"].dt.day
    dim["DayName"] = dim["Date"].dt.strftime("%A")
    dim["WeekdayNum"] = dim["Date"].dt.weekday + 1
    dim["IsWeekend"] = dim["WeekdayNum"].isin([6, 7])

    return dim[["DateKey", "Date", "Year", "Quarter", "QuarterName", "Month",
                "MonthName", "MonthShort", "YearMonth", "Day", "DayName",
                "WeekdayNum", "IsWeekend"]]

In [16]:
def build_fact_sales(sales: pd.DataFrame) -> pd.DataFrame:
    fact = sales[["ProductKey", "CustomerKey", "OrderDate", "Quantity", "Net Price"]].copy()
    fact["DateKey"] = fact["OrderDate"].dt.strftime("%Y%m%d").astype(int)
    fact["SalesAmount"] = (fact["Quantity"] * fact["Net Price"]).round(2)
    fact = fact.rename(columns={"Net Price": "NetPrice"})
    fact.insert(0, "SalesKey", range(1, len(fact) + 1))
    return fact[["SalesKey", "DateKey", "ProductKey", "CustomerKey",
                 "Quantity", "NetPrice", "SalesAmount"]]


def build_fact_forecast(forecast: pd.DataFrame,
                        dim_brand: pd.DataFrame,
                        dim_geo: pd.DataFrame) -> pd.DataFrame:
    fact = forecast.merge(dim_brand, on="Brand", how="left")
    fact = fact.merge(dim_geo[["GeographyKey", "CountryRegion"]], on="CountryRegion", how="left")
    fact = fact[["Year", "GeographyKey", "BrandKey", "Forecast"]].copy()
    fact.insert(0, "ForecastKey", range(1, len(fact) + 1))
    return fact

## 5. Validate and Load
Run referential checks and write output CSV files.

In [17]:
def validate(fact_sales: pd.DataFrame,
             fact_forecast: pd.DataFrame,
             dim_product: pd.DataFrame,
             dim_customer: pd.DataFrame,
             dim_date: pd.DataFrame,
             dim_brand: pd.DataFrame,
             dim_geo: pd.DataFrame):
    errors = []

    orphan_products = ~fact_sales["ProductKey"].isin(dim_product["ProductKey"])
    if orphan_products.any():
        errors.append(f"FactSales has {orphan_products.sum()} rows with orphan ProductKey")

    orphan_customers = ~fact_sales["CustomerKey"].isin(dim_customer["CustomerKey"])
    if orphan_customers.any():
        errors.append(f"FactSales has {orphan_customers.sum()} rows with orphan CustomerKey")

    orphan_dates = ~fact_sales["DateKey"].isin(dim_date["DateKey"])
    if orphan_dates.any():
        errors.append(f"FactSales has {orphan_dates.sum()} rows with orphan DateKey")

    orphan_fc_brand = fact_forecast["BrandKey"].isna().sum()
    if orphan_fc_brand:
        errors.append(f"FactForecast has {orphan_fc_brand} rows with unmatched BrandKey")

    orphan_fc_geo = fact_forecast["GeographyKey"].isna().sum()
    if orphan_fc_geo:
        errors.append(f"FactForecast has {orphan_fc_geo} rows with unmatched GeographyKey")

    if (fact_sales["SalesAmount"] <= 0).any():
        errors.append("FactSales contains non-positive SalesAmount values")

    if errors:
        for e in errors:
            log.error(f"VALIDATE | FAIL - {e}")
        raise ValueError("Validation failed - see log for details")

    log.info("VALIDATE | All referential integrity and range checks PASSED")


def load(tables: dict):
    log.info("LOAD | Writing output CSV files")
    for name, df in tables.items():
        path = OUT_DIR / f"{name}.csv"
        df.to_csv(path, index=False)
        log.info(f"LOAD | wrote {path} ({len(df):,} rows, {len(df.columns)} cols)")

## 6. Run ETL
Execute the full pipeline from raw JSON to CSV outputs.

In [18]:
raw_sales = extract(SALES_FILE,"Sales.json")
raw_forecast = extract(FORECAST_FILE,"Forecast.json")

sales = clean_sales(raw_sales)
forecast = clean_forecast(raw_forecast)

dim_product = build_dim_product(sales)
dim_customer = build_dim_customer(sales)
dim_geo = build_dim_geography(sales, forecast)
dim_brand = build_dim_brand(sales, forecast)
dim_date = build_dim_date(sales, forecast)

fact_sales = build_fact_sales(sales)
fact_forecast = build_fact_forecast(forecast, dim_brand, dim_geo)

validate(fact_sales, fact_forecast, dim_product, dim_customer, dim_date, dim_brand, dim_geo)

load({
    "dim_date": dim_date,
    "dim_product": dim_product,
    "dim_customer": dim_customer,
    "dim_geography": dim_geo,
    "dim_brand": dim_brand,
    "fact_sales": fact_sales,
    "fact_forecast": fact_forecast,
})

2026-06-13 14:06:17,239 | INFO     | EXTRACT | Loading Sales.json from C:\Users\ahmed.deabes\Downloads\orion-sales-analytics\src\Data\Raw\Sales.json
2026-06-13 14:06:19,479 | INFO     | EXTRACT | Sales.json: 298,246 rows, 18 columns
2026-06-13 14:06:19,698 | INFO     | EXTRACT | Loading Forecast.json from C:\Users\ahmed.deabes\Downloads\orion-sales-analytics\src\Data\Raw\forecast.json
2026-06-13 14:06:19,701 | INFO     | EXTRACT | Forecast.json: 33 rows, 4 columns
2026-06-13 14:06:19,703 | INFO     | CLEAN | Sales - starting cleaning routine
2026-06-13 14:06:20,756 | INFO     | CLEAN | Sales - removed 218,008 duplicate rows (298,246 -> 80,238)
2026-06-13 14:06:21,278 | INFO     | CLEAN | Sales - trimmed whitespace on 14 text columns
2026-06-13 14:06:21,348 | INFO     | CLEAN | Sales - dropped redundant 'Color' column
2026-06-13 14:06:21,391 | INFO     | CLEAN | Sales - filled 50,441 nulls in 'Name' with 'Unknown'
2026-06-13 14:06:21,405 | INFO     | CLEAN | Sales - filled 50,441 nulls 